# 🩸 JORINOVA NEXUS — Leukaemia / blast-cell detector (fine-tuning)

Fine-tunes a **YOLOv8** detector to find **blast cells** on a blood/marrow smear and
produces `leukemia.pt`. Blasts are flagged **critical** and mapped to a leukaemia type.

> **Fine-tuning, NOT from scratch** — starts from pretrained weights
> (`yolov8s.pt`, or your `malaria.pt`) and adapts them.

**Before you start:** Runtime → **Change runtime type** → **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics roboflow kaggle
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Get the project code + mount Drive

In [ ]:
import os
from getpass import getpass
REPO = 'https://github.com/dujoely1/JORINOVA-NEXUS.git'
tok = getpass('GitHub token (press Enter if the repo is public): ').strip()
url = REPO.replace('https://', f'https://{tok}@') if tok else REPO
os.system(f'rm -rf /content/nexus && git clone --depth 1 {url} /content/nexus')
print('repo cloned:', os.path.exists('/content/nexus/ml/leukemia'))
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_leukemia', exist_ok=True)
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data — Option A (Roboflow, ready) or Option B (Kaggle C-NMC)

### ✅ Option A — Roboflow ALL (acute lymphoblastic leukaemia) detector — **ready, no blanks**
Concrete project **`lucas-reis/acute-lymphoblastic-leukemia-dataset`** (object detection, benign vs malignant blasts). **Not Kaggle** — uses your free Roboflow key (Settings → API). The cell auto-picks a version and **remaps the classes to the app's keys** (`normal`, `lymphoblast`) so the trained model plugs straight into `leukemia_disorders.json`. Run it, then go to step 5.

*Fallback projects if that one is unavailable: search https://universe.roboflow.com/search?q=leukemia%20blast and use `show download code` (workspace/project/version).*

In [ ]:
# ── Option A: Roboflow ALL detector -> classes remapped to app keys (no blanks) ──
import os, glob, yaml
from getpass import getpass
os.environ['ROBOFLOW_API_KEY'] = getpass('Roboflow Private API Key: ').strip()
try:
    from roboflow import Roboflow
except ModuleNotFoundError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','roboflow'], check=True)
    from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])

# Concrete ALL detection dataset (not Kaggle). Auto-picks an available version.
proj = rf.workspace('lucas-reis').project('acute-lymphoblastic-leukemia-dataset')
ds = None
for v in range(1, 12):
    try:
        ds = proj.version(v).download('yolov8', location='/content/data/leuk_rf'); print('✓ downloaded v', v); break
    except Exception:
        continue
assert ds, 'No version downloaded — use a fallback project (see the markdown above).'

# --- remap the dataset's class names -> leukemia_disorders.json keys ---
def to_appkey(name):
    n = str(name).strip().lower()
    if any(k in n for k in ('benign', 'hem', 'normal', 'healthy', 'non')): return 'normal'
    if 'myelo' in n:                                                        return 'myeloblast'
    if any(k in n for k in ('malign','blast','all','pre-b','pre_b','pro-b','pro_b','early','leuk','lympho')):
        return 'lymphoblast'
    return 'normal'

dyaml = os.path.join(ds.location, 'data.yaml')
d = yaml.safe_load(open(dyaml)); names = d['names']
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
mapped = [to_appkey(n) for n in names]
uniq = []
for a in mapped:
    if a not in uniq: uniq.append(a)                 # final class list (stable order)
remap = {i: uniq.index(a) for i, a in enumerate(mapped)}
print('class map:', dict(zip(names, mapped)), '-> final', uniq)

# rewrite label class indices to the collapsed app classes
for lbl in glob.glob(ds.location + '/**/labels/*.txt', recursive=True):
    out = []
    for ln in open(lbl):
        p = ln.split()
        if p: p[0] = str(remap[int(p[0])]); out.append(' '.join(p))
    open(lbl, 'w').write(chr(10).join(out))

d['names'] = uniq; d['nc'] = len(uniq)
yaml.safe_dump(d, open(dyaml, 'w'), sort_keys=False)
DATA_YAML = dyaml
print('✓ DATA_YAML =', DATA_YAML, '| classes:', uniq)

### Option B — Kaggle C-NMC 2019 (ALL vs normal, single cells → detection boxes)
C-NMC is a CLASSIFICATION set (one centred cell per image). We wrap each cell in a full-
image box so YOLO can train a blast-vs-normal detector. Needs `kaggle.json`.

In [ ]:
# 1) upload kaggle.json, then download + unzip (slug may differ — search kaggle 'C-NMC leukemia')
# from google.colab import files; files.upload()   # pick kaggle.json
# import os; os.makedirs('/root/.kaggle', exist_ok=True)
# os.replace('kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
# !kaggle datasets download -d avk256/cnmc-leukemia -p /content/data/cnmc --unzip
#
# 2) wrap single-cell images as full-image boxes -> YOLO (all=lymphoblast, hem/normal=normal)
import os, glob, shutil
from pathlib import Path
from PIL import Image
base = '/content/data/cnmc'
CLASSES = ['normal', 'lymphoblast']   # names match leukemia_disorders.json
def label_of(path):
    p = path.lower()
    if os.sep + 'all' in p or '_all' in p or 'lymphoblast' in p or 'leukem' in p: return 1
    if 'hem' in p or 'normal' in p or 'benign' in p: return 0
    return None
imgs = [p for ext in ('*.bmp','*.jpg','*.jpeg','*.png','*.tiff') for p in glob.glob(base + '/**/' + ext, recursive=True)]
assert imgs, 'No images found under ' + base + ' — check the download/slug.'
OUT = '/content/data/leuk_yolo'
nval = max(1, len(imgs)//7)
for k, src in enumerate(imgs):
    lab = label_of(src)
    if lab is None: continue
    split = 'val' if k < nval else 'train'
    oi = Path(OUT)/'images'/split; ol = Path(OUT)/'labels'/split
    oi.mkdir(parents=True, exist_ok=True); ol.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, oi/os.path.basename(src))
    # full-image bbox (cell fills the frame in C-NMC/ALL-IDB2 crops)
    (ol/(Path(src).stem + '.txt')).write_text(f'{lab} 0.5 0.5 0.98 0.98')
DATA_YAML = os.path.join(OUT, 'data.yaml')
open(DATA_YAML, 'w').write(f'path: {OUT}\ntrain: images/train\nval: images/val\nnc: {len(CLASSES)}\nnames: {CLASSES}\n')
print('C-NMC ->', DATA_YAML, '| classes:', CLASSES)

## 5. Choose the base weights — fine-tuning (never from scratch)

In [ ]:
import os
USE_MALARIA_BASE = False   # True -> transfer from your malaria detector (microscopy features)
malaria_pt = '/content/nexus/backend/models/malaria/malaria.pt'
BASE_WEIGHTS = malaria_pt if (USE_MALARIA_BASE and os.path.exists(malaria_pt)) else 'yolov8s.pt'
print('fine-tuning from:', BASE_WEIGHTS)

## 6. Fine-tune the detector

In [ ]:
EPOCHS = 100
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
import os
PROJECT = '/content/runs/detect'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/nexus_leukemia/runs'; os.makedirs(PROJECT, exist_ok=True)
    print('runs -> Drive:', PROJECT)
except Exception as e:
    print('Drive not mounted, runs stay on /content:', e)
model = YOLO(BASE_WEIGHTS)   # pretrained -> fine-tune; head adapts to blast/normal classes
results = model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16,
                      project=PROJECT, name='leukemia', pretrained=True, patience=25, plots=True,
                      hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=180,
                      fliplr=0.5, flipud=0.5, mosaic=1.0, translate=0.1, scale=0.5)
m = model.val()
print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')

## 7. Export `leukemia.pt` (app path + Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/nexus_leukemia'; os.makedirs(DRIVE, exist_ok=True)
except Exception as e:
    print('Drive not mounted (continuing):', e)
best_hits = [c for c in glob.glob('/content/**/weights/best.pt', recursive=True) if os.path.isfile(c)]
last_hits = [c for c in glob.glob('/content/**/weights/last.pt', recursive=True) if os.path.isfile(c)]
backup    = [c for c in glob.glob((DRIVE or '') + '/**/*.pt', recursive=True) if DRIVE and os.path.isfile(c)]
pool = best_hits or last_hits or backup
if not pool:
    raise FileNotFoundError('No trained weights found — run step 6 (training) first.')
best = sorted(pool, key=os.path.getmtime)[-1]
print('using weights:', best, '->', 'best.pt' if best_hits else ('last.pt' if last_hits else 'Drive backup'))
os.makedirs('/content/nexus/backend/models/leukemia', exist_ok=True)
dest = '/content/nexus/backend/models/leukemia/leukemia.pt'; shutil.copy(best, dest); print('saved to app path:', dest)
if DRIVE:
    shutil.copy(best, os.path.join(DRIVE, 'leukemia.pt')); print('backed up to Drive')
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
try:
    YOLO(best).export(format='onnx', opset=12)
except Exception as e:
    print('onnx export skipped:', e)
from google.colab import files
files.download(best)

## 8. Put it in the app
1. Move `leukemia.pt` → **`backend/models/leukemia/leukemia.pt`**, commit, push.
2. Auto-loads for `image_type` ∈ {`leukemia`, `blast`}.
3. On Render, build with `INSTALL_ML=1` (≥2 GB) to run the detector; else Claude vision reads the field.

Detected blasts → leukaemia type + **critical** flag via
`backend/ai_services/leukemia_disorders.json` (blast → acute leukaemia; Auer rod → AML).